# AIAgent execution-value explorer

Vary the execution parameters and re-run the regime-conditioned execution against the AI agent's actual trades (or a synthetic series). All logic lives in `src/order_mgmt/agent/`; this notebook is a thin control surface.

**What is measured:** *implementation shortfall* (realized price vs the agent's arrival price, in ticks). There is **no order-book data**, so this is not market-impact slippage. The fill model is optimistic (touching the limit = fill). The **median** is the honest headline — the mean is dragged by the unfilled-chase tail. See `notes/agent-value.md`.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from IPython.display import display

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from order_mgmt.agent.loader import find_agent_csv, find_market_dir, load_agent_series, load_market_for_agent
from order_mgmt.agent.metrics import evaluate_agent_execution
from order_mgmt.agent.synthetic import synth_agent_series

SEED = 0  # reproducibility

In [ ]:
market_w = widgets.Dropdown(options=["Gold", "Nasdaq", "EuroStoxx", "Bunds", "GBP", "JPY", "HeatingOil"], value="Gold", description="Market")
tau_w = widgets.Dropdown(options=[5, 10, 15, 30, 60], value=5, description="τ (min)")
hl_w = widgets.IntSlider(value=20, min=5, max=60, step=5, description="half-life m")
M_w = widgets.IntSlider(value=3, min=1, max=5, description="M (vol)")
N_w = widgets.IntSlider(value=3, min=1, max=5, description="N (range)")
K_w = widgets.IntSlider(value=3, min=1, max=5, description="K (Δx)")
frt_w = widgets.FloatSlider(value=0.6, min=0.1, max=0.95, step=0.05, description="fill target")
synth_w = widgets.Checkbox(value=True, description="overlay synthetic")
run_btn = widgets.Button(description="Run evaluation", button_style="primary")
out = widgets.Output()


def _run(_=None):
    out.clear_output(wait=True)
    with out:
        market_dir = find_market_dir(ROOT / "data", market_w.value)
        if market_dir is None:
            print(f"market dir for {market_w.value!r} not found")
            return
        df_ohlcv, tick, proper_days, stem = load_market_for_agent(market_dir)
        kw = dict(tick=tick, proper_days=proper_days, tau=tau_w.value, half_life=hl_w.value,
                  M=M_w.value, N=N_w.value, K=K_w.value, fill_rate_target=frt_w.value, j_start=200)
        results = {}
        agent_csv = find_agent_csv(market_dir)
        if agent_csv is not None:
            results["real"] = evaluate_agent_execution(load_agent_series(agent_csv, market_dir.name), df_ohlcv, **kw)
        if synth_w.value:
            results["synthetic"] = evaluate_agent_execution(synth_agent_series(df_ohlcv, seed=SEED, n_decisions=500), df_ohlcv, **kw)
        fig, ax = plt.subplots(figsize=(8, 4.5))
        for name, r in results.items():
            vals = [f.shortfall_ticks for f in r.fills]
            if not vals:
                continue
            print(f"{name:10} n={r.n_decisions} fill={r.fill_rate:.1%} mean={r.mean_shortfall_ticks:+.2f}t median={r.median_shortfall_ticks:+.2f}t vs_vwap={r.value_add_vs_vwap_ticks:+.2f}t unfilled-tail={r.unfilled_tail_cost_ticks:+.2f}t")
            ax.hist(vals, bins=50, alpha=0.5, label=f"{name} (median {np.median(vals):+.1f}t)")
        ax.axvline(0, color="black", ls="--", lw=0.7)
        ax.set_title(f"{market_dir.name} — implementation shortfall (ticks; +ve = beat arrival)")
        ax.set_xlabel("ticks"); ax.set_ylabel("count"); ax.legend()
        plt.show()


run_btn.on_click(_run)
display(widgets.VBox([widgets.HBox([market_w, tau_w, hl_w]), widgets.HBox([M_w, N_w, K_w]), widgets.HBox([frt_w, synth_w, run_btn]), out]))
_run()